# The Mathematics of LoRA and QLoRA

This notebook derives, from scratch, why the adapters attached in
`src/model/lora_config.py` work, and where every number that script
prints (trainable parameter count, memory footprint) actually comes
from. Read this before trusting the implementation - the code is the
executable version of what follows, not the other way around.

References: Hu et al. 2021, *LoRA: Low-Rank Adaptation of Large
Language Models*; Dettmers et al. 2023, *QLoRA: Efficient Finetuning of
Quantized LLMs*.

## 1. The problem: full fine-tuning is too expensive

A single linear layer inside a transformer block computes
$$h = W_0 x, \qquad W_0 \in \mathbb{R}^{d \times k}.$$

Full fine-tuning updates every entry of $W_0$: $d \cdot k$ trainable
parameters for *that one layer*. Mistral-7B has $7.24$ billion
parameters spread across 32 transformer blocks, each with 7 such linear
projections (`q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj,
down_proj`). Full fine-tuning means:

- optimizer states (Adam: 2 extra fp32 buffers per parameter) for all 7.24B params
- gradients for all 7.24B params
- the params themselves

At fp16 params + fp32 Adam states this is well over 60GB - completely
infeasible on an 8GB card. We need a method that touches only a tiny
fraction of the parameters.

## 2. The low-rank hypothesis

Hu et al. observed empirically that the *weight update* $\Delta W$
learned during fine-tuning tends to have low **intrinsic rank** - most
of what changes during adaptation lives in a small subspace, even
though $W_0$ itself is full-rank. This motivates replacing the update
with an explicit low-rank factorization:

$$\Delta W = BA, \qquad B \in \mathbb{R}^{d \times r},\ A \in \mathbb{R}^{r \times k},\ r \ll \min(d, k)$$

Instead of learning $d \cdot k$ numbers directly, we learn $B$ and $A$:
$r(d + k)$ numbers. $W_0$ itself is **frozen** - never updated, never
even given a gradient buffer.

In [ ]:
import sympy as sp

d, k, r = sp.symbols('d k r', positive=True, integer=True)

full_finetune_params = d * k
lora_params = r * (d + k)
reduction_ratio = sp.simplify(lora_params / full_finetune_params)

print('Full fine-tuning parameters per layer: d*k =', full_finetune_params)
print('LoRA parameters per layer:          r*(d+k) =', lora_params)
print('Ratio LoRA/full (symbolic):', reduction_ratio)

Now substitute Mistral-7B-Instruct-v0.3's actual dimensions, read
straight from the model config (no hand-typed numbers to get wrong) -
this assumes you already ran `src/model/base_model.py` once, so the
config is cached locally and this does not need the full 14GB of
weights, just the small `config.json`.

In [ ]:
from transformers import AutoConfig

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'
cfg = AutoConfig.from_pretrained(MODEL_NAME)

hidden = cfg.hidden_size
intermediate = cfg.intermediate_size
n_layers = cfg.num_hidden_layers
n_kv_heads = cfg.num_key_value_heads
n_heads = cfg.num_attention_heads
head_dim = hidden // n_heads
kv_dim = n_kv_heads * head_dim  # k_proj/v_proj are smaller under grouped-query attention

print(f'hidden_size={hidden}, intermediate_size={intermediate}, n_layers={n_layers}')
print(f'n_heads={n_heads}, n_kv_heads={n_kv_heads} (GQA), head_dim={head_dim}, kv_dim={kv_dim}')

In [ ]:
# Per-layer (d, k) for each of the 7 target modules LoRA is attached to.
# Attention projections use kv_dim for k_proj/v_proj because Mistral uses
# grouped-query attention (fewer KV heads than query heads) - this is
# exactly the kind of detail that is easy to get wrong by hand and easy
# to get right by reading it from the config, which is why we did that above.
shapes = {
    'q_proj':    (hidden, hidden),
    'k_proj':    (kv_dim, hidden),
    'v_proj':    (kv_dim, hidden),
    'o_proj':    (hidden, hidden),
    'gate_proj': (intermediate, hidden),
    'up_proj':   (intermediate, hidden),
    'down_proj': (hidden, intermediate),
}

R = 16  # matches DEFAULT_R in src/model/lora_config.py

full_total = 0
lora_total = 0
for name, (d_, k_) in shapes.items():
    full = d_ * k_
    lora = R * (d_ + k_)
    full_total += full
    lora_total += lora
    print(f'{name:10s} d={d_:6d} k={k_:6d}  full={full:9,d}  lora(r={R})={lora:7,d}')

full_total_all_layers = full_total * n_layers
lora_total_all_layers = lora_total * n_layers

print()
print(f'Summed over all {n_layers} layers:')
print(f'  Full fine-tuning would touch: {full_total_all_layers:,} params')
print(f'  LoRA (r={R}) touches:          {lora_total_all_layers:,} params')
print(f'  Ratio: {100*lora_total_all_layers/full_total_all_layers:.3f}%')
print()
print('This lora_total_all_layers number should match, almost exactly, the')
print('"Trainable params" count printed by running:')
print('    python src\\model\\lora_config.py')
print('(small difference only if the LoRA bias mode or embedding config differs).')

## 3. The forward pass and the scaling factor

With the adapter in place, the layer's forward pass becomes
$$h = W_0 x + \Delta W x = W_0 x + BA x = W_0 x + \frac{\alpha}{r} B (A x)$$

Two implementation details worth naming explicitly, both visible in
`build_lora_config()`:

- $Ax$ is computed **before** multiplying by $B$: this costs
  $O(rk) + O(dr)$ multiply-adds instead of $O(dk)$ for a dense update -
  the low-rank structure saves compute, not just parameters.
- The $\alpha / r$ scaling term (`lora_alpha / r`) rescales the update
  so that changing $r$ (e.g. going from rank 16 to rank 32) does not
  silently change the effective learning rate on $\Delta W$. With
  `DEFAULT_ALPHA = 32` and `DEFAULT_R = 16`, the scale factor is exactly
  $2.0$.

## 4. Where the gradients go

Because $W_0$ is frozen, backpropagation only needs $\partial L / \partial A$
and $\partial L / \partial B$. Let $g = \partial L / \partial h$ be the
upstream gradient arriving at this layer's output, and drop the
$\alpha/r$ scale for a moment (it just multiplies both results
linearly). Since $h = W_0 x + B(Ax)$:

$$\frac{\partial L}{\partial B} = g \, (Ax)^\top, \qquad
  \frac{\partial L}{\partial A} = B^\top g \, x^\top$$

Notice $W_0$ never appears in either expression - it genuinely receives
zero gradient, which is exactly why `prepare_model_for_kbit_training`
and `get_peft_model` can leave it in 4-bit, non-differentiable form
the whole time. Let's verify this derivation numerically rather than
just trust the algebra.

In [ ]:
import torch

torch.manual_seed(0)
d_, k_, r_ = 32, 24, 4

W0 = torch.randn(d_, k_, requires_grad=False)  # frozen, no grad needed
A = torch.randn(r_, k_, requires_grad=True)
B = torch.randn(d_, r_, requires_grad=True)
x = torch.randn(k_, 1)
target = torch.randn(d_, 1)

h = W0 @ x + B @ (A @ x)
loss = ((h - target) ** 2).sum()
loss.backward()

g = 2 * (h - target).detach()  # dL/dh for a sum-of-squares loss, shape (d_, 1)

manual_grad_B = g @ (A @ x).detach().T
manual_grad_A = (B.detach().T @ g) @ x.T

print('grad_B matches autograd:', torch.allclose(manual_grad_B, B.grad, atol=1e-5))
print('grad_A matches autograd:', torch.allclose(manual_grad_A, A.grad, atol=1e-5))
print('W0.grad is None (frozen, never touched):', W0.grad is None)

## 5. Initialization: why the adapter starts as a no-op

`peft` initializes $A$ with small random Gaussian entries and $B$ as an
**all-zero matrix**. Since $\Delta W = BA$, this means $\Delta W = 0$ at
the very start of training:

$$h_{\text{step }0} = W_0 x + \frac{\alpha}{r}\, \underbrace{B}_{=0} (Ax) = W_0 x$$

The adapted model is *mathematically identical* to the base model
before any training happens. This matters for a fine-tuning project
specifically: whatever generation quality `base_model.py`'s smoke test
showed, attaching a freshly-initialized LoRA adapter (as
`lora_config.py` does) cannot make it worse - training can only move
away from that starting point.

## 6. QLoRA: quantizing $W_0$ itself

Everything above assumes $W_0$ is just "frozen" - QLoRA goes further
and compresses $W_0$ to 4 bits per weight, which is what
`build_quant_config()` in `base_model.py` sets up. Three pieces:

**NF4 (NormalFloat4).** Pretrained transformer weights are
empirically close to $\mathcal{N}(0, \sigma^2)$. A quantization grid
with *uniformly spaced* levels wastes resolution in the sparse tails
and under-resolves the dense center. NF4 instead places its 16
representable values at the quantiles of a standard normal distribution
- each of the 16 values represents an equal amount of *probability
mass* of the input distribution, not an equal amount of the numeric
range. This is why NF4 outperforms plain int4 or fp4 at the same bit
width for this specific use case.

**Block-wise quantization.** Weights are quantized in small contiguous
blocks (blocksize 64 by default), each with its own scale constant $c$:
$$W_{4bit} = \text{round}\!\left(\frac{W}{c}\right), \qquad
  \hat{W} = c \cdot \text{dequant}(W_{4bit})$$
This keeps outlier weights in one block from blowing up the resolution
available to every other block.

**Double quantization** (`bnb_4bit_use_double_quant=True`). Storing one
fp32 scale constant $c$ per 64-weight block is itself $32/64 = 0.5$
bits/parameter of overhead. Double quantization quantizes *those
constants* too (to 8-bit, in blocks of 256), cutting that overhead to
roughly $0.5/256 \times 8 + \ldots \approx 0.127$ bits/parameter - a
detail, but on 7 billion parameters it is not a small one.

In [ ]:
n_params = sum(p.numel() for p in [torch.empty(1)])  # placeholder, replaced below
n_params = 7_240_000_000  # Mistral-7B-Instruct-v0.3 parameter count

def gib(bits_per_param):
    return n_params * bits_per_param / 8 / (1024**3)

print(f'{"fp32":8s}: {gib(32):6.2f} GiB')
print(f'{"fp16/bf16":8s}: {gib(16):6.2f} GiB')
print(f'{"int8":8s}: {gib(8):6.2f} GiB')
print(f'{"NF4 only":8s}: {gib(4):6.2f} GiB')
print(f'{"NF4 + double-quant":8s}: {gib(4 + 0.127):6.2f} GiB  <- what base_model.py actually loads')
print()
print('This is the number to compare against the report_gpu_memory("after load")')
print('line that src/model/base_model.py prints when you run it - expect it to')
print('be somewhat higher than the raw weight size above, because activations,')
print('the KV cache, and CUDA/cuDNN workspace memory add on top of the weights.')

## 7. Merging back into a single weight matrix

Once training (Step 4) is done, `src/training/merge.py` will compute
$$W' = W_0 + \frac{\alpha}{r} BA$$
and write $W'$ out as an ordinary dense weight matrix. This matters
operationally: a merged model has **no LoRA-related inference overhead
at all** - it is indistinguishable in structure from a normally
fine-tuned model, and can be served without `peft` installed, without
the base/adapter split, and without the small forward-pass latency
cost of computing $B(Ax)$ as a separate term at serving time.

## Summary

- Full fine-tuning of one linear layer costs $d \cdot k$ parameters;
  LoRA replaces the update with $\Delta W = BA$, costing $r(d+k)$ -
  verified above to be under 1% of Mistral-7B's parameters at $r=16$.
- $W_0$ receives no gradient ever (proven numerically above), which is
  exactly what lets it be stored in a non-differentiable 4-bit format.
- $B$ starts at zero, so the adapted model equals the base model before
  training starts.
- QLoRA's NF4 + double quantization compresses $W_0$ to roughly
  4.13 bits/parameter, matching the memory numbers `base_model.py`
  reports at runtime.

Everything in `src/model/base_model.py` and `src/model/lora_config.py`
is a direct implementation of the equations above - if a number printed
by those scripts does not match a number computed in this notebook,
that is a bug to chase down, not a coincidence to ignore.